# Prompt 7: Complex Types and Collections in Terraform

**Exam Objective 4 — Terraform Configuration (HCL)**  
**HashiCorp Certified: Terraform Associate (004)**

---

## Topics Covered

| Topic | Section |
|---|---|
| Primitive types | 1 |
| Collection types: list, map, set | 2 |
| Structural types: object, tuple | 3 |
| `any` type | 4 |
| Type conversion functions | 5 |
| `count` meta-argument | 6 |
| `for_each` meta-argument | 7 |
| `for` expressions | 8 |
| Splat expressions | 9 |
| `dynamic` block | 10 |
| Exam-style questions | 11 |
| Real-world scenarios | 12 |
| Quick-reference cheat sheet | 13 |

---
## 1. Primitive Types

Terraform has three primitive types — the building blocks of all other types.

| Type | Description | Example Value |
|---|---|---|
| `string` | Sequence of Unicode characters | `"us-east-1"` |
| `number` | Numeric value (integer or float) | `3`, `3.14` |
| `bool` | Boolean true or false | `true`, `false` |

### Implicit Type Conversions

Terraform will automatically convert between primitive types in many contexts:

```hcl
# string "1" is automatically coerced to number 1 when used in a numeric context
variable "count_str" {
  type    = string
  default = "3"
}

resource "random_pet" "example" {
  count = var.count_str   # Terraform coerces "3" → 3
}
```

```hcl
# bool true/false ↔ string "true"/"false" coercion
variable "enabled" {
  type    = bool
  default = true
}

output "as_string" {
  value = tostring(var.enabled)  # "true"
}
```

### Key Rules
- Terraform is **strongly typed** at runtime — you cannot mix incompatible types without explicit conversion.
- Implicit coercion only works between the three primitive types, not between collections.
- If a variable is declared `type = string` and you pass a number, Terraform converts it for you — but the reverse must sometimes be explicit.

---
## 2. Collection Types

Collection types group multiple values of the **same element type** together.

---

### 2.1 `list(type)` — Ordered Sequence

- Elements are ordered and accessed by **zero-based integer index**.
- Duplicate values are allowed.
- All elements must be the same type (or coercible to it).

```hcl
variable "availability_zones" {
  type    = list(string)
  default = ["us-east-1a", "us-east-1b", "us-east-1c"]
}

# Access by index
output "first_az" {
  value = var.availability_zones[0]   # "us-east-1a"
}

output "last_az" {
  value = var.availability_zones[length(var.availability_zones) - 1]  # "us-east-1c"
}
```

**Use cases:**
- Ordered resource lists (subnets in sequence)
- Items where positional meaning matters
- Inputs to `count`-based iteration

---

### 2.2 `map(type)` — Key-Value Pairs

- Unordered key-value store where **all values** must be the same type.
- Keys are always `string`; values are the declared type.
- Access by key using bracket notation or `lookup()`.

```hcl
variable "tags" {
  type = map(string)
  default = {
    Environment = "production"
    Team        = "platform"
    CostCenter  = "cc-1234"
  }
}

# Access by key
output "env_tag" {
  value = var.tags["Environment"]  # "production"
}

# Safe access with default using lookup()
output "owner_tag" {
  value = lookup(var.tags, "Owner", "unknown")  # "unknown" — key not present
}
```

**Use cases:**
- Resource tag maps
- Environment-specific configuration (AMI IDs per region)
- `for_each` iteration over named resources

---

### 2.3 `set(type)` — Unordered Unique Values

- **No duplicate values** — adding a duplicate silently drops it.
- **No index access** — sets have no defined order, so `var.my_set[0]` is an **error**.
- Ideal for `for_each` iteration where order doesn't matter but uniqueness does.

```hcl
variable "allowed_cidrs" {
  type    = set(string)
  default = ["10.0.0.0/8", "172.16.0.0/12", "192.168.0.0/16"]
}

# ERROR — sets cannot be indexed
# output "first" { value = var.allowed_cidrs[0] }  # causes error

# CORRECT — convert to list first (order not guaranteed)
output "first_cidr" {
  value = tolist(var.allowed_cidrs)[0]
}

# Check membership
output "contains_private" {
  value = contains(var.allowed_cidrs, "10.0.0.0/8")  # true
}
```

**Use cases:**
- `for_each` on resource keys
- Deduplication of values
- IAM permission sets, security group CIDR lists

---

### Collection Type Comparison

```
┌──────────────┬───────────┬────────────┬────────────────┬───────────────────┐
│ Type         │ Ordered?  │ Indexed?   │ Duplicates?    │ Best use case     │
├──────────────┼───────────┼────────────┼────────────────┼───────────────────┤
│ list(T)      │ Yes       │ [N]        │ Yes            │ Ordered sequences │
│ map(T)       │ No        │ ["key"]    │ No (keys)      │ Named k/v pairs   │
│ set(T)       │ No        │ ✗          │ No             │ Unique values     │
└──────────────┴───────────┴────────────┴────────────────┴───────────────────┘
```

---
## 3. Structural Types

Structural types group multiple values where **each element can have a different type**.

---

### 3.1 `object({...})` — Named Attributes with Mixed Types

An `object` is like a map, but each key (attribute) is declared with its own specific type.

```hcl
variable "server_config" {
  type = object({
    instance_type = string
    disk_size_gb  = number
    enable_backup = bool
    tags          = map(string)
  })
  default = {
    instance_type = "t3.medium"
    disk_size_gb  = 100
    enable_backup = true
    tags          = { Environment = "prod" }
  }
}

# Attribute access
output "instance_type" {
  value = var.server_config.instance_type   # "t3.medium"
}

output "disk_gb" {
  value = var.server_config.disk_size_gb    # 100
}
```

**Key behaviour:** An `object` type can have **optional** attributes using `optional()` (Terraform 1.3+):

```hcl
variable "database" {
  type = object({
    engine         = string
    engine_version = optional(string, "8.0")   # default if not provided
    port           = optional(number, 3306)
  })
}
```

---

### 3.2 `tuple([...])` — Positional Elements with Mixed Types

A `tuple` is like a list, but each position has its own declared type. Access by **integer index**.

```hcl
variable "record" {
  type    = tuple([string, number, bool])
  default = ["web-server", 8080, true]
}

output "name"    { value = var.record[0] }   # "web-server"
output "port"    { value = var.record[1] }   # 8080
output "enabled" { value = var.record[2] }   # true
```

**When to use tuple vs list:**
- Use `list(T)` when all elements are the same type and the list can vary in length.
- Use `tuple([T1, T2, ...])` when the number of elements is fixed and each has a specific type.
- Tuples are uncommon in practice — mainly used when returning multiple values of mixed types from modules.

---

### Object vs Map — Key Distinction

```
┌───────────────┬─────────────────────────────────┬─────────────────────────────────┐
│ Feature       │ map(T)                          │ object({k = T, ...})            │
├───────────────┼─────────────────────────────────┼─────────────────────────────────┤
│ Value types   │ All values SAME type            │ Each attribute its OWN type     │
│ Keys          │ Any string key allowed          │ Keys defined at declaration     │
│ Optional keys │ Any key can be absent           │ Use optional() for optional     │
│ Access        │ var.m["key"] or var.m.key       │ var.o.attribute_name            │
│ Use case      │ Tags, AMI IDs by region         │ Complex structured config       │
└───────────────┴─────────────────────────────────┴─────────────────────────────────┘
```

---
## 4. The `any` Type

The `any` type is a **special placeholder** that disables static type checking for a variable. Terraform infers the actual type from the value provided at runtime.

```hcl
variable "flexible_input" {
  type    = any
  default = "a string, but could be a number or list"
}
```

### When `any` is Useful

- Module variables that must accept different value shapes depending on the caller.
- Variables that are passed through without being inspected by the module.
- Rapid prototyping when you haven't finalised the type shape.

### When NOT to Use `any`

```hcl
# AVOID: no type safety, no validation possible
variable "config" {
  type = any
}

# PREFER: explicit type enables validation, IDE hints, and clear errors
variable "config" {
  type = object({
    region      = string
    instance_type = string
  })
}
```

**Exam rule of thumb:** `any` disables type checking — use sparingly and only when a more specific type is genuinely not possible.

---
## 5. Type Conversion Functions

Terraform provides explicit conversion functions to move between compatible collection types.

### `tolist(value)` — Convert to List

```hcl
# Convert a set to a list (order not guaranteed)
variable "cidrs" {
  type    = set(string)
  default = ["10.0.1.0/24", "10.0.2.0/24"]
}

locals {
  cidr_list = tolist(var.cidrs)
  # cidr_list[0] is now accessible — but order is not deterministic
}
```

### `toset(value)` — Convert to Set (Deduplicate)

```hcl
locals {
  raw_list = ["web", "db", "web", "cache"]  # has duplicate
  unique   = toset(local.raw_list)           # {"web", "db", "cache"} — deduped
}

# Useful to make a list safe for for_each:
resource "aws_subnet" "by_tier" {
  for_each = toset(["web", "app", "data"])
  # ...
}
```

### `tomap(value)` — Convert to Map

```hcl
locals {
  as_map = tomap({
    name = "my-server"
    type = "t3.micro"
  })
}
```

### Primitive Conversion Functions

```hcl
tostring(42)       # "42"
tostring(true)     # "true"
tonumber("3.14")   # 3.14
tobool("true")     # true
tobool("false")    # false
```

### Conversion Function Summary

```
┌─────────────┬────────────────────┬──────────────────────────────────────────┐
│ Function    │ Input → Output     │ Common use case                          │
├─────────────┼────────────────────┼──────────────────────────────────────────┤
│ tolist()    │ set/tuple → list   │ Enable index access on a set             │
│ toset()     │ list → set         │ Deduplicate; enable for_each on a list   │
│ tomap()     │ object → map       │ Treat fixed-key object as a map          │
│ tostring()  │ any → string       │ Output/logging of non-string values      │
│ tonumber()  │ string/bool → num  │ Math on string-typed numeric inputs      │
│ tobool()    │ string → bool      │ Parse "true"/"false" strings             │
└─────────────┴────────────────────┴──────────────────────────────────────────┘
```

---
## 6. The `count` Meta-Argument

`count` is a meta-argument available on every `resource` and `module` block. It creates **N identical copies** of the resource, differentiated by a numeric index.

### Basic Syntax

```hcl
resource "aws_instance" "web" {
  count         = 3
  ami           = "ami-0abcdef1234567890"
  instance_type = "t3.micro"

  tags = {
    Name = "web-${count.index}"   # web-0, web-1, web-2
  }
}
```

### `count.index`

- Available inside any resource block that uses `count`.
- Zero-based integer: `0`, `1`, `2`, ...
- Use it to create unique names, pick from a list, or vary configuration.

```hcl
variable "subnet_ids" {
  type    = list(string)
  default = ["subnet-aaa", "subnet-bbb", "subnet-ccc"]
}

resource "aws_instance" "web" {
  count     = length(var.subnet_ids)
  subnet_id = var.subnet_ids[count.index]   # distributes instances across subnets
  # ...
}
```

### Resource Addressing with `count`

```
aws_instance.web[0]    # first instance
aws_instance.web[1]    # second instance
aws_instance.web[*].id # list of all instance IDs (splat)
```

### ⚠️ The `count` Order Sensitivity Problem

```hcl
# If you start with this:
variable "names" {
  default = ["alice", "bob", "carol"]
}
resource "aws_iam_user" "users" {
  count = length(var.names)
  name  = var.names[count.index]
}
# Resources: aws_iam_user.users[0]=alice, [1]=bob, [2]=carol

# Then remove "bob" from the middle:
variable "names" {
  default = ["alice", "carol"]   # bob removed
}
# Terraform sees: [0]=alice (no change), [1]=carol (was bob → REPLACE), [2] DESTROY
# Alice is unchanged but carol gets replaced — potentially destructive!
```

**Exam rule:** Use `count` for truly identical resources where order doesn't matter. Use `for_each` when resources have meaningful names and you need to safely add/remove individual items.

---
## 7. The `for_each` Meta-Argument

`for_each` creates one resource instance **per element** of a `map` or `set`. Each instance is keyed by the element's key or value — making it robust to insertions and deletions.

### `for_each` with a `map`

```hcl
variable "buckets" {
  type = map(object({
    region      = string
    versioning  = bool
  }))
  default = {
    assets   = { region = "us-east-1", versioning = false }
    backups  = { region = "us-west-2", versioning = true  }
    logs     = { region = "eu-west-1", versioning = false }
  }
}

resource "aws_s3_bucket" "all" {
  for_each = var.buckets

  bucket = "my-company-${each.key}"           # each.key = "assets", "backups", "logs"
  region = each.value.region                  # each.value = the object for this entry

  versioning {
    enabled = each.value.versioning
  }
}
```

### `for_each` with a `set`

```hcl
resource "aws_security_group_rule" "allow_ports" {
  for_each  = toset(["80", "443", "8080"])

  type      = "ingress"
  from_port = tonumber(each.key)   # each.key = each.value for sets
  to_port   = tonumber(each.key)
  protocol  = "tcp"
  # ...
}
```

### `each.key` and `each.value`

| Iteration over | `each.key` | `each.value` |
|---|---|---|
| `map(T)` | The map key (string) | The map value (type T) |
| `set(T)` | The set element (same as value) | The set element (same as key) |

### Resource Addressing with `for_each`

```
aws_s3_bucket.all["assets"]   # the assets bucket
aws_s3_bucket.all["backups"]  # the backups bucket
```

### `count` vs `for_each` — Decision Guide

```
┌────────────────────────────────┬──────────────┬──────────────┐
│ Scenario                       │ count        │ for_each     │
├────────────────────────────────┼──────────────┼──────────────┤
│ N identical resources          │ ✅ Good      │ Overkill     │
│ Resources with unique names    │ ⚠️  Risky    │ ✅ Preferred │
│ Safe add/remove individual     │ ❌ Reorders  │ ✅ Safe      │
│ Accepts map or set input       │ ❌ No        │ ✅ Yes       │
│ Accepts list input             │ ✅ Yes       │ ⚠️  toset()  │
│ Index access in expressions    │ [N]          │ ["key"]      │
│ Terraform state key format     │ resource[0]  │ resource["k"]│
└────────────────────────────────┴──────────────┴──────────────┘
```

---
## 8. `for` Expressions

`for` expressions transform one collection into another — similar to list/dict comprehensions in Python.

### List Comprehension — `[for ...]`

```hcl
variable "names" {
  type    = list(string)
  default = ["alice", "bob", "carol"]
}

locals {
  # Transform each element
  upper_names = [for name in var.names : upper(name)]
  # Result: ["ALICE", "BOB", "CAROL"]

  # Add index to each element
  indexed_names = [for i, name in var.names : "${i}: ${name}"]
  # Result: ["0: alice", "1: bob", "2: carol"]

  # Filter with if clause
  long_names = [for name in var.names : name if length(name) > 3]
  # Result: ["alice", "carol"]  ("bob" excluded — length 3, not > 3)
}
```

### Map Comprehension — `{for ...}`

```hcl
variable "instance_ids" {
  type = map(string)
  default = {
    web = "i-0001"
    app = "i-0002"
    db  = "i-0003"
  }
}

locals {
  # Invert key and value
  inverted = {for k, v in var.instance_ids : v => k}
  # Result: {"i-0001" = "web", "i-0002" = "app", "i-0003" = "db"}

  # Add prefix to all keys
  prefixed = {for k, v in var.instance_ids : "server-${k}" => v}
  # Result: {"server-web" = "i-0001", ...}

  # Filter map entries
  non_db = {for k, v in var.instance_ids : k => v if k != "db"}
  # Result: {"web" = "i-0001", "app" = "i-0002"}
}
```

### Iterating over a List to Produce a Map

```hcl
variable "users" {
  type = list(object({
    name  = string
    email = string
  }))
  default = [
    { name = "alice", email = "alice@example.com" },
    { name = "bob",   email = "bob@example.com"   },
  ]
}

locals {
  # Build a map keyed by user name
  users_by_name = {for user in var.users : user.name => user.email}
  # Result: {"alice" = "alice@example.com", "bob" = "bob@example.com"}
}
```

### `for` Expression Syntax Summary

```
List:  [for <element>            in <collection>        : <transform>  [if <condition>]]
List:  [for <index>, <element>   in <collection>        : <transform>  [if <condition>]]
Map:   {for <key>, <value>       in <map_or_object>     : <k> => <v>   [if <condition>]}
Map:   {for <element>            in <list_or_set>       : <k> => <v>   [if <condition>]}
```

---
## 9. Splat Expressions

Splat expressions (`[*]`) are shorthand for extracting a single attribute from every element of a `list`, `tuple`, or `set` produced by `count`-based resources.

### Legacy Splat (`[*]`) — Works with count

```hcl
resource "aws_instance" "web" {
  count         = 3
  ami           = "ami-0abcdef1234567890"
  instance_type = "t3.micro"
}

output "all_ids" {
  # Equivalent to: [for i in aws_instance.web : i.id]
  value = aws_instance.web[*].id
  # Result: ["i-001", "i-002", "i-003"]
}

output "all_private_ips" {
  value = aws_instance.web[*].private_ip
}
```

### Full Splat (`[*]`) vs Attribute-only Splat (`.*.`)

```hcl
# Full splat — applied to a list/tuple of objects
var.servers[*].name         # extracts .name from every element

# Equivalent for expression
[for s in var.servers : s.name]
```

### ⚠️ Splat Does NOT Work with `for_each`

```hcl
# for_each produces a map — splat does not apply to maps
resource "aws_s3_bucket" "all" {
  for_each = var.bucket_names
  # ...
}

# ERROR — [*] does not work on for_each resources (they're maps, not lists)
# output "ids" { value = aws_s3_bucket.all[*].id }

# CORRECT — use a for expression with values()
output "ids" {
  value = [for b in aws_s3_bucket.all : b.id]
}
```

**Exam key point:** Splat expressions work on **lists/tuples** (from `count`). For `for_each` resources, use a `for` expression with `values()` instead.

---
## 10. The `dynamic` Block

A `dynamic` block generates **repeated nested configuration blocks** programmatically from a collection. This avoids copy-pasting identical nested blocks.

### Problem Without `dynamic`

```hcl
# Repetitive — adding each port rule as a separate block
resource "aws_security_group" "web" {
  name = "web-sg"

  ingress {
    from_port   = 80
    to_port     = 80
    protocol    = "tcp"
    cidr_blocks = ["0.0.0.0/0"]
  }

  ingress {
    from_port   = 443
    to_port     = 443
    protocol    = "tcp"
    cidr_blocks = ["0.0.0.0/0"]
  }

  ingress {
    from_port   = 8080
    to_port     = 8080
    protocol    = "tcp"
    cidr_blocks = ["10.0.0.0/8"]
  }
}
```

### Solution With `dynamic`

```hcl
variable "ingress_rules" {
  type = list(object({
    port        = number
    cidr_blocks = list(string)
  }))
  default = [
    { port = 80,   cidr_blocks = ["0.0.0.0/0"]  },
    { port = 443,  cidr_blocks = ["0.0.0.0/0"]  },
    { port = 8080, cidr_blocks = ["10.0.0.0/8"] },
  ]
}

resource "aws_security_group" "web" {
  name = "web-sg"

  dynamic "ingress" {
    for_each = var.ingress_rules          # iterate over the list
    content {                             # content = what each block will contain
      from_port   = ingress.value.port
      to_port     = ingress.value.port
      protocol    = "tcp"
      cidr_blocks = ingress.value.cidr_blocks
    }
  }
}
```

### `dynamic` Block Syntax Rules

```
dynamic "<block_type>" {
  for_each = <collection>
  iterator = <optional_name>   # defaults to the block_type label

  content {
    # Use <block_type>.key   and   <block_type>.value
    # OR  <iterator>.key     and   <iterator>.value   if iterator is set
  }
}
```

### Using the `iterator` Argument

```hcl
dynamic "ingress" {
  for_each = var.ingress_rules
  iterator = rule                  # rename loop variable from "ingress" to "rule"

  content {
    from_port   = rule.value.port  # rule.value instead of ingress.value
    to_port     = rule.value.port
    protocol    = "tcp"
    cidr_blocks = rule.value.cidr_blocks
  }
}
```

### Nested `dynamic` Blocks

```hcl
resource "aws_elastic_beanstalk_environment" "app" {
  # ...
  dynamic "setting" {
    for_each = var.settings
    content {
      namespace = setting.value.namespace
      name      = setting.value.name
      value     = setting.value.value
    }
  }
}
```

**Exam key point:** A `dynamic` block only works for **nested blocks** (like `ingress`, `egress`, `setting`) — it cannot generate top-level resource arguments like `ami` or `instance_type`.

---
## 11. Exam-Style Practice Questions

---

**Q1: A Terraform configuration declares a variable of type `set(string)` containing values `["web", "app", "db"]`. Which of the following operations will cause a Terraform error?**

A) `for_each = var.tiers` — using the variable directly in a `for_each`  
B) `length(var.tiers)` — getting the element count  
C) `var.tiers[0]` — accessing the first element by index  
D) `contains(var.tiers, "web")` — checking membership  

<details>
<summary>Answer</summary>

**Answer: C**  
Sets are **unordered collections** — they have no positional index, so `var.tiers[0]` is an error in Terraform. Option A is valid — `for_each` accepts sets directly. Option B is valid — `length()` works on all collection types. Option D is valid — `contains()` works on sets. To access a specific element from a set by position, first convert it with `tolist(var.tiers)[0]`, but be aware the order is not guaranteed.

</details>

---

**Q2: A configuration uses `count = 3` to create three identically-configured EC2 instances. A later change removes the middle instance from the list used to drive count. What is the primary risk of this change compared to using `for_each`?**

A) `count` is deprecated in Terraform and will be removed in a future version  
B) Removing a middle element causes Terraform to destroy and recreate all resources with a higher index, even if their configuration hasn't changed  
C) `count` cannot be used with EC2 instances — only with modules  
D) The removed resource's state is retained indefinitely, causing a drift error on every subsequent `terraform plan`  

<details>
<summary>Answer</summary>

**Answer: B**  
This is the core weakness of `count` — resources are addressed by **numeric index**. When an element is removed from the middle, all subsequent resources shift down one index. Terraform sees the shifted resources as having changed (they now have a different index) and plans to destroy and recreate them, even if only one resource was intended to be removed. `for_each` avoids this because resources are keyed by a stable string identifier. Option A is false — `count` is not deprecated. Option C is false. Option D is false.

</details>

---

**Q3: Which of the following `for` expressions produces a map from a list of objects?**

```hcl
variable "servers" {
  default = [
    { name = "web", ip = "10.0.0.1" },
    { name = "db",  ip = "10.0.0.2" },
  ]
}
```

A) `[for s in var.servers : s.name => s.ip]`  
B) `{for s in var.servers : s.name => s.ip}`  
C) `for s in var.servers : { s.name = s.ip }`  
D) `map(var.servers, s => s.name)`  

<details>
<summary>Answer</summary>

**Answer: B**  
A `for` expression wrapped in `{}` curly braces produces a **map**. The syntax `{for s in collection : key_expr => value_expr}` produces `{"web" = "10.0.0.1", "db" = "10.0.0.2"}`. Option A uses `[]` square brackets — that produces a **list**, and `s.name => s.ip` is invalid list syntax. Option C is not valid HCL — `for` inside bare braces is not valid. Option D is not a real Terraform function.

</details>

---

**Q4: A developer wants to generate a repeated nested `ingress` block inside an `aws_security_group` resource based on a variable list. Which Terraform construct should they use?**

A) A `for` expression in the `ingress` block argument  
B) A `count` meta-argument on the `ingress` block  
C) A `dynamic "ingress"` block with a `content` sub-block and `for_each` argument  
D) A `locals` block that generates multiple `ingress` definitions  

<details>
<summary>Answer</summary>

**Answer: C**  
The `dynamic` block is specifically designed for generating repeated **nested blocks** based on a collection. Its syntax is `dynamic "<block_label>" { for_each = <collection>  content { ... } }`. Option A is incorrect — `for` expressions produce values (strings, lists, maps), not block structures. Option B is incorrect — `count` is not a valid argument inside nested blocks like `ingress`. Option D is incorrect — `locals` compute values but cannot generate block structures.

</details>

---
## 12. Real-World Scenarios

<details>
<summary>Scenario 1: Migrating from count to for_each Without Destroying Resources</summary>

**Situation:**  
A team manages 4 IAM users with `count`. They want to migrate to `for_each` without destroying and recreating users.

**Original configuration:**
```hcl
variable "usernames" {
  default = ["alice", "bob", "carol", "dave"]
}

resource "aws_iam_user" "users" {
  count = length(var.usernames)
  name  = var.usernames[count.index]
}
# State: aws_iam_user.users[0], [1], [2], [3]
```

**Target configuration:**
```hcl
variable "usernames" {
  type    = set(string)
  default = ["alice", "bob", "carol", "dave"]
}

resource "aws_iam_user" "users" {
  for_each = var.usernames
  name     = each.key
}
# State will be: aws_iam_user.users["alice"], ["bob"], ["carol"], ["dave"]
```

**Solution — use `moved` blocks to re-key without destroying:**
```hcl
moved {
  from = aws_iam_user.users[0]
  to   = aws_iam_user.users["alice"]
}
moved {
  from = aws_iam_user.users[1]
  to   = aws_iam_user.users["bob"]
}
moved {
  from = aws_iam_user.users[2]
  to   = aws_iam_user.users["carol"]
}
moved {
  from = aws_iam_user.users[3]
  to   = aws_iam_user.users["dave"]
}
```

**Expected outcome:** `terraform plan` shows the moves but no `+`/`-` — resources are just re-keyed in state with zero infrastructure changes.  
**Exam sub-objective:** Objective 4 — `count` vs `for_each` trade-offs; `moved` block usage.

</details>

<details>
<summary>Scenario 2: Building a Dynamic Security Group with a Variable Rule List</summary>

**Situation:**  
A security team wants to manage firewall rules as a structured variable so rules can be added/removed without touching the resource block.

```hcl
variable "sg_rules" {
  type = list(object({
    description = string
    port        = number
    protocol    = string
    cidr_blocks = list(string)
  }))
  default = [
    { description = "HTTP",  port = 80,  protocol = "tcp", cidr_blocks = ["0.0.0.0/0"] },
    { description = "HTTPS", port = 443, protocol = "tcp", cidr_blocks = ["0.0.0.0/0"] },
    { description = "SSH",   port = 22,  protocol = "tcp", cidr_blocks = ["10.0.0.0/8"] },
  ]
}

resource "aws_security_group" "app" {
  name   = "app-sg"
  vpc_id = var.vpc_id

  dynamic "ingress" {
    for_each = var.sg_rules
    content {
      description = ingress.value.description
      from_port   = ingress.value.port
      to_port     = ingress.value.port
      protocol    = ingress.value.protocol
      cidr_blocks = ingress.value.cidr_blocks
    }
  }

  egress {
    from_port   = 0
    to_port     = 0
    protocol    = "-1"
    cidr_blocks = ["0.0.0.0/0"]
  }
}
```

**Expected outcome:** Adding or removing a rule from `sg_rules` in `terraform.tfvars` is the only change needed — the resource block itself never changes.  
**Exam sub-objective:** Objective 4 — `dynamic` block for repeated nested blocks.

</details>

<details>
<summary>Scenario 3: Multi-Region Resource Tagging with for Expressions</summary>

**Situation:**  
A team needs to create S3 buckets in 3 regions and tag each bucket with its region and environment.

```hcl
variable "regions" {
  type    = list(string)
  default = ["us-east-1", "us-west-2", "eu-west-1"]
}

variable "environment" {
  type    = string
  default = "production"
}

locals {
  # Build a map of region -> bucket config using a for expression
  bucket_configs = {
    for region in var.regions : region => {
      bucket_name = "my-app-${var.environment}-${replace(region, "-", "")}"
      tags = {
        Region      = region
        Environment = var.environment
      }
    }
  }
}

# Would require provider aliases for each region in a real setup
# Shown here to demonstrate the for expression producing the map
output "bucket_configs" {
  value = local.bucket_configs
}
# Output:
# {
#   "us-east-1" = { bucket_name = "my-app-production-useast1", tags = {...} }
#   "us-west-2" = { bucket_name = "my-app-production-uswest2", tags = {...} }
#   ...
# }
```

**Expected outcome:** The `for` expression produces a structured map that can be fed to `for_each`, keeping configuration DRY.  
**Exam sub-objective:** Objective 4 — `for` expressions producing maps.

</details>

<details>
<summary>Scenario 4: Safely Deduplicating a List Input with toset()</summary>

**Situation:**  
An engineer receives a `list(string)` variable from callers that may contain duplicate values. Using it directly with `for_each` will error because `for_each` requires unique keys.

```hcl
variable "env_names" {
  type        = list(string)
  description = "Caller may pass duplicates"
  default     = ["dev", "staging", "dev", "production", "staging"]
}

resource "aws_ssm_parameter" "env_flag" {
  # ERROR if env_names has duplicates:
  # for_each = var.env_names   # requires set or map — list has duplicates

  # CORRECT — toset() deduplicates:
  for_each = toset(var.env_names)

  name  = "/app/environments/${each.key}"
  type  = "String"
  value = each.key
}
# Creates 3 parameters: /app/environments/dev, /staging, /production
```

**CLI check:**
```bash
terraform plan
# Plan: 3 to add  (not 5 — duplicates removed by toset)
```

**Expected outcome:** `toset()` silently deduplicates, ensuring `for_each` receives a valid set.  
**Exam sub-objective:** Objective 4 — type conversion functions; `for_each` requirements.

</details>

<details>
<summary>Scenario 5: Extracting All Output Values from for_each Resources</summary>

**Situation:**  
A team creates multiple S3 buckets with `for_each` and wants to output all bucket ARNs as a list for use by another system.

```hcl
variable "bucket_names" {
  type    = set(string)
  default = ["assets", "backups", "logs"]
}

resource "aws_s3_bucket" "store" {
  for_each = var.bucket_names
  bucket   = "my-company-${each.key}"
}

# WRONG — splat does not work on for_each resources (they are maps, not lists)
# output "arns" { value = aws_s3_bucket.store[*].arn }  # ERROR

# CORRECT option 1 — for expression with values()
output "bucket_arns" {
  value = [for b in aws_s3_bucket.store : b.arn]
}

# CORRECT option 2 — produce a map keyed by bucket name
output "bucket_arn_map" {
  value = {for k, b in aws_s3_bucket.store : k => b.arn}
}
```

**Expected outcome:**
```
bucket_arns = [
  "arn:aws:s3:::my-company-assets",
  "arn:aws:s3:::my-company-backups",
  "arn:aws:s3:::my-company-logs",
]
```

**Exam sub-objective:** Objective 4 — splat limitations with `for_each`; `for` expressions with resource maps.

</details>

---
## 13. Quick-Reference Cheat Sheet

```
╔══════════════════════════════════════════════════════════════════════════════╗
║               TERRAFORM TYPES & COLLECTIONS — EXAM CHEAT SHEET             ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ PRIMITIVE TYPES                                                             ║
║  string  number  bool                                                       ║
║  Implicit coercion between primitives allowed                               ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ COLLECTION TYPES — all elements same type                                   ║
║  list(T)  → ordered, indexed [N], duplicates OK                             ║
║  map(T)   → unordered, keyed ["key"], no duplicate keys                     ║
║  set(T)   → unordered, NO index access, no duplicates                       ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ STRUCTURAL TYPES — each attribute its own type                              ║
║  object({attr = type, ...})  → access via .attribute                        ║
║  tuple([T1, T2, ...])        → fixed-length, access by index                ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ any TYPE                                                                    ║
║  Disables type checking — avoid unless necessary                            ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ CONVERSION FUNCTIONS                                                        ║
║  tolist()  toset()  tomap()  tostring()  tonumber()  tobool()               ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ count          → N identical resources; count.index; address: resource[N]   ║
║ for_each       → per map/set element; each.key each.value; resource["key"]  ║
║ count problem  → removing middle element reorders/recreates higher indexes  ║
║ for_each wins  → named resources, safe add/remove, stable keys              ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ for EXPRESSIONS                                                             ║
║  [for x in list : transform(x) if condition]     → list                     ║
║  {for k, v in map : new_k => new_v if condition} → map                      ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ SPLAT  resource[*].attr  → list of attrs from count resources               ║
║        Does NOT work on for_each (use for expression instead)               ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ dynamic BLOCK — generates repeated NESTED blocks (not top-level args)       ║
║  dynamic "block" { for_each = col  content { block.value.attr } }           ║
╚══════════════════════════════════════════════════════════════════════════════╝
```